# 2-D Microfluidic Protein Transport (Cartesian Coordinates - Unsteady State)

## ChBE 3300: Multidimensional Fluids and Heat Transport

### Problem Description

This notebook simulates **unsteady-state fluid flow and protein transport** in a microfluidic T-junction mixer, commonly used in:
- Protein crystallization studies
- Drug delivery research
- Biomolecular mixing and reaction analysis
- Lab-on-a-chip devices

### Physical System

A microfluidic channel where:
- **Inlet 1** (left): Buffer solution (no protein)
- **Inlet 2** (top): Protein solution (high concentration)
- **Outlet** (right): Mixed stream

The flow is laminar (low Reynolds number, Re ~ 1-10) as typical in microfluidics.

### Governing Equations

#### 1. Continuity Equation (Incompressible Flow)
$$\frac{\partial u}{\partial x} + \frac{\partial v}{\partial y} = 0$$

#### 2. Navier-Stokes Equations (Unsteady State, Cartesian Coordinates)

**x-momentum:**
$$\rho\left(\frac{\partial u}{\partial t} + u\frac{\partial u}{\partial x} + v\frac{\partial u}{\partial y}\right) = -\frac{\partial p}{\partial x} + \mu\left(\frac{\partial^2 u}{\partial x^2} + \frac{\partial^2 u}{\partial y^2}\right)$$

**y-momentum:**
$$\rho\left(\frac{\partial v}{\partial t} + u\frac{\partial v}{\partial x} + v\frac{\partial v}{\partial y}\right) = -\frac{\partial p}{\partial y} + \mu\left(\frac{\partial^2 v}{\partial x^2} + \frac{\partial^2 v}{\partial y^2}\right)$$

#### 3. Protein Transport Equation (Advection-Diffusion)
$$\frac{\partial C}{\partial t} + u\frac{\partial C}{\partial x} + v\frac{\partial C}{\partial y} = D\left(\frac{\partial^2 C}{\partial x^2} + \frac{\partial^2 C}{\partial y^2}\right)$$

where:
- $u, v$ = velocity components in x and y directions (m/s)
- $p$ = pressure (Pa)
- $\rho$ = fluid density (kg/m³)
- $\mu$ = dynamic viscosity (Pa·s)
- $C$ = protein concentration (mol/m³)
- $D$ = protein diffusion coefficient (m²/s)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import cm
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
import warnings
warnings.filterwarnings('ignore')

%matplotlib inline

# Set plotting style
plt.rcParams['figure.dpi'] = 100
plt.rcParams['font.size'] = 10

## 1. Physical Parameters and Grid Setup

In [ ]:
# Physical parameters (typical for aqueous protein solutions)
rho = 1000.0          # Density (kg/m³) - water
mu = 0.001            # Dynamic viscosity (Pa·s) - water at 20°C
nu = mu / rho         # Kinematic viscosity (m²/s)
D_protein = 1e-10     # Protein diffusion coefficient (m²/s) - typical for small protein
C0 = 1.0              # Initial protein concentration in inlet (mol/m³)

# Channel geometry
Lx = 200e-6           # Channel length (200 μm)
Ly = 100e-6           # Channel width (100 μm)
inlet_width = 40e-6   # Width of side inlet (40 μm)

# Flow parameters
U_inlet = 1e-3        # Inlet velocity (1 mm/s - typical microfluidic velocity)
Re = rho * U_inlet * Ly / mu  # Reynolds number
Pe = U_inlet * Ly / D_protein  # Péclet number (advection/diffusion)

print("=" * 60)
print("MICROFLUIDIC PROTEIN TRANSPORT SIMULATION")
print("=" * 60)
print(f"\nPhysical Parameters:")
print(f"  Density: {rho} kg/m³")
print(f"  Viscosity: {mu*1000:.2f} mPa·s")
print(f"  Protein diffusion coefficient: {D_protein:.2e} m²/s")
print(f"\nChannel Dimensions:")
print(f"  Length: {Lx*1e6:.0f} μm")
print(f"  Width: {Ly*1e6:.0f} μm")
print(f"\nDimensionless Numbers:")
print(f"  Reynolds number (Re): {Re:.2f}")
print(f"  Péclet number (Pe): {Pe:.2e}")
print(f"\nFlow Regime: {'Laminar' if Re < 2300 else 'Turbulent'}")
print(f"Transport: {'Diffusion-dominated' if Pe < 1 else 'Advection-dominated'}")
print("=" * 60)

In [ ]:
# Grid parameters
nx = 80               # Number of grid points in x
ny = 40               # Number of grid points in y
dx = Lx / (nx - 1)    # Grid spacing in x
dy = Ly / (ny - 1)    # Grid spacing in y

# Time parameters
dt = 0.001            # Time step (s)
nt = 500              # Number of time steps
t_total = dt * nt     # Total simulation time

# Stability criteria
CFL = U_inlet * dt / min(dx, dy)  # CFL condition
diff_limit = nu * dt / min(dx, dy)**2  # Diffusion stability

print(f"\nGrid Setup:")
print(f"  Grid points: {nx} × {ny}")
print(f"  Grid spacing: Δx = {dx*1e6:.2f} μm, Δy = {dy*1e6:.2f} μm")
print(f"  Time step: {dt} s")
print(f"  Total time: {t_total:.3f} s")
print(f"\nStability Check:")
print(f"  CFL number: {CFL:.4f} (should be < 1.0)")
print(f"  Diffusion limit: {diff_limit:.4f} (should be < 0.5)")
print(f"  Status: {'✓ STABLE' if CFL < 1.0 and diff_limit < 0.5 else '✗ UNSTABLE - Reduce dt'}")

## 2. Initialize Flow and Concentration Fields

In [ ]:
# Create coordinate grids
x = np.linspace(0, Lx, nx)
y = np.linspace(0, Ly, ny)
X, Y = np.meshgrid(x, y)

# Initialize fields
u = np.zeros((ny, nx))       # x-velocity (m/s)
v = np.zeros((ny, nx))       # y-velocity (m/s)
p = np.zeros((ny, nx))       # pressure (Pa)
C = np.zeros((ny, nx))       # protein concentration (mol/m³)

# Storage for time evolution
u_new = u.copy()
v_new = v.copy()
C_new = C.copy()

print("Fields initialized successfully!")
print(f"Array shapes: u{u.shape}, v{v.shape}, C{C.shape}")

## 3. Simplified Flow Solution

For this educational example, we'll use a simplified approach:
- Assume fully developed Poiseuille flow in x-direction
- Parabolic velocity profile in y-direction
- Focus on the advection-diffusion of protein

This simplification is valid for:
- Long, straight channels
- Low Reynolds number flow
- Development length << channel length

In [ ]:
# Create parabolic velocity profile (Poiseuille flow)
# u(y) = U_max * (1 - (2y/H - 1)^2) where U_max = 1.5 * U_avg
U_max = 1.5 * U_inlet

for j in range(ny):
    y_normalized = (y[j] - Ly/2) / (Ly/2)  # Normalize from -1 to 1
    u[j, :] = U_max * (1 - y_normalized**2)  # Parabolic profile

# v-velocity remains zero (fully developed flow)
v[:, :] = 0

# Set initial protein concentration in top half (simulating top inlet)
inlet_region = int(0.6 * ny)  # Top 40% of channel
C[inlet_region:, 0:5] = C0    # Protein enters from top-left region

print(f"Flow field initialized:")
print(f"  Maximum velocity: {U_max*1000:.3f} mm/s")
print(f"  Average velocity: {U_inlet*1000:.3f} mm/s")
print(f"  Residence time: {Lx/U_inlet:.3f} s")

## 4. Time Evolution - Protein Transport

Solve the advection-diffusion equation using explicit finite differences:

$$C_{i,j}^{n+1} = C_{i,j}^n - \frac{u\Delta t}{\Delta x}(C_{i,j}^n - C_{i-1,j}^n) - \frac{v\Delta t}{\Delta y}(C_{i,j}^n - C_{i,j-1}^n)$$
$$+ \frac{D\Delta t}{\Delta x^2}(C_{i+1,j}^n - 2C_{i,j}^n + C_{i-1,j}^n) + \frac{D\Delta t}{\Delta y^2}(C_{i,j+1}^n - 2C_{i,j}^n + C_{i,j-1}^n)$$

In [ ]:
# Store snapshots for visualization
snapshot_times = [0, 100, 200, 300, 400, 499]
C_snapshots = []
time_labels = []

# Time integration
print("\nStarting time integration...")
print("Progress: ", end="")

for n in range(nt):
    # Progress indicator
    if n % 100 == 0:
        print(f"{n}/{nt} ", end="", flush=True)
    
    # Store snapshots
    if n in snapshot_times:
        C_snapshots.append(C.copy())
        time_labels.append(f"t = {n*dt:.3f} s")
    
    # Update concentration using upwind scheme for advection
    C_new = C.copy()
    
    for i in range(1, ny-1):
        for j in range(1, nx-1):
            # Advection terms (upwind differencing)
            if u[i, j] > 0:
                dC_dx = (C[i, j] - C[i, j-1]) / dx
            else:
                dC_dx = (C[i, j+1] - C[i, j]) / dx
            
            if v[i, j] > 0:
                dC_dy = (C[i, j] - C[i-1, j]) / dy
            else:
                dC_dy = (C[i+1, j] - C[i, j]) / dy
            
            # Diffusion terms (central differencing)
            d2C_dx2 = (C[i, j+1] - 2*C[i, j] + C[i, j-1]) / dx**2
            d2C_dy2 = (C[i+1, j] - 2*C[i, j] + C[i-1, j]) / dy**2
            
            # Update equation
            C_new[i, j] = (C[i, j] - dt * (u[i, j] * dC_dx + v[i, j] * dC_dy) +
                          dt * D_protein * (d2C_dx2 + d2C_dy2))
    
    # Boundary conditions
    # Inlet: protein concentration in top portion
    C_new[inlet_region:, 0] = C0
    C_new[0:inlet_region, 0] = 0
    
    # Outlet: zero gradient (convective outflow)
    C_new[:, -1] = C_new[:, -2]
    
    # Walls: no flux
    C_new[0, :] = C_new[1, :]
    C_new[-1, :] = C_new[-2, :]
    
    # Update
    C = C_new.copy()

print(f"{nt}/{nt}")
print("\nTime integration complete!")
print(f"Final maximum concentration: {np.max(C):.4f} mol/m³")
print(f"Final average concentration: {np.mean(C):.4f} mol/m³")

## 5. Visualization of Results

In [ ]:
# Convert to micrometers for plotting
X_um = X * 1e6
Y_um = Y * 1e6
x_um = x * 1e6
y_um = y * 1e6

# Create comprehensive visualization
fig = plt.figure(figsize=(16, 10))
gs = fig.add_gridspec(3, 3, hspace=0.3, wspace=0.3)

# Time evolution snapshots
for idx, (C_snap, time_label) in enumerate(zip(C_snapshots, time_labels)):
    ax = fig.add_subplot(gs[idx // 3, idx % 3])
    
    # Plot concentration field
    im = ax.contourf(X_um, Y_um, C_snap, levels=20, cmap='YlOrRd')
    
    # Add velocity vectors (subsample for clarity)
    skip = 4
    ax.quiver(X_um[::skip, ::skip], Y_um[::skip, ::skip],
             u[::skip, ::skip], v[::skip, ::skip],
             alpha=0.3, scale=0.05, width=0.003)
    
    ax.set_xlabel('x (μm)')
    ax.set_ylabel('y (μm)')
    ax.set_title(time_label)
    ax.set_aspect('equal')
    plt.colorbar(im, ax=ax, label='C (mol/m³)')

plt.suptitle('Protein Transport in Microfluidic Channel - Time Evolution', 
             fontsize=14, fontweight='bold', y=0.98)
plt.show()

In [ ]:
# Detailed analysis at final time
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Concentration field with streamlines
ax = axes[0, 0]
contour = ax.contourf(X_um, Y_um, C, levels=30, cmap='viridis')
# Create streamplot
speed = np.sqrt(u**2 + v**2)
ax.streamplot(x_um, y_um, u.T, v.T, color='white', 
              density=1.5, linewidth=1, arrowsize=1.5)
plt.colorbar(contour, ax=ax, label='Concentration (mol/m³)')
ax.set_xlabel('x (μm)')
ax.set_ylabel('y (μm)')
ax.set_title('Final Concentration Field with Flow Streamlines')
ax.set_aspect('equal')

# 2. Velocity profile
ax = axes[0, 1]
ax.plot(u[:, nx//2] * 1000, y_um, 'b-', linewidth=2, label='Mid-channel')
ax.plot(u[:, 3*nx//4] * 1000, y_um, 'r--', linewidth=2, label='Downstream')
ax.set_xlabel('Velocity (mm/s)')
ax.set_ylabel('y (μm)')
ax.set_title('Velocity Profiles (Poiseuille Flow)')
ax.legend()
ax.grid(True, alpha=0.3)

# 3. Concentration profiles at different x-locations
ax = axes[1, 0]
x_positions = [nx//4, nx//2, 3*nx//4]
x_labels = ['25%', '50%', '75%']
colors = ['blue', 'green', 'red']

for x_pos, x_label, color in zip(x_positions, x_labels, colors):
    ax.plot(C[:, x_pos], y_um, linewidth=2, label=f'x = {x_label} L', color=color)

ax.set_xlabel('Concentration (mol/m³)')
ax.set_ylabel('y (μm)')
ax.set_title('Protein Concentration Profiles Across Channel')
ax.legend()
ax.grid(True, alpha=0.3)

# 4. Centerline concentration evolution
ax = axes[1, 1]
centerline_idx = ny // 2
ax.plot(x_um, C[centerline_idx, :], 'b-', linewidth=2, label='Centerline')
ax.plot(x_um, C[3*ny//4, :], 'r--', linewidth=2, label='Upper region')
ax.plot(x_um, C[ny//4, :], 'g:', linewidth=2, label='Lower region')
ax.set_xlabel('x (μm)')
ax.set_ylabel('Concentration (mol/m³)')
ax.set_title('Axial Concentration Distribution')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 6. Mixing Efficiency Analysis

In [ ]:
# Calculate mixing index at outlet
# Mixing index: 1 - σ/σ_max, where σ is standard deviation

outlet_C = C[:, -1]
C_ideal = C0 / 2  # Ideal completely mixed concentration
sigma = np.std(outlet_C)
sigma_max = C0 / 2  # Maximum possible std dev
mixing_index = 1 - sigma / sigma_max

print("\n" + "="*60)
print("MIXING EFFICIENCY ANALYSIS")
print("="*60)
print(f"Outlet concentration statistics:")
print(f"  Mean: {np.mean(outlet_C):.4f} mol/m³")
print(f"  Std Dev: {sigma:.4f} mol/m³")
print(f"  Min: {np.min(outlet_C):.4f} mol/m³")
print(f"  Max: {np.max(outlet_C):.4f} mol/m³")
print(f"\nMixing Index: {mixing_index:.2%}")
print(f"  (0% = no mixing, 100% = perfect mixing)")
print("="*60)

# Visualization of outlet profile
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(outlet_C, y_um, 'b-', linewidth=2.5)
ax1.axvline(C_ideal, color='r', linestyle='--', linewidth=2, label='Ideal mixed')
ax1.fill_betweenx(y_um, 0, outlet_C, alpha=0.3)
ax1.set_xlabel('Concentration (mol/m³)', fontsize=12)
ax1.set_ylabel('y (μm)', fontsize=12)
ax1.set_title('Outlet Concentration Profile', fontsize=13, fontweight='bold')
ax1.legend(fontsize=11)
ax1.grid(True, alpha=0.3)

# Histogram
ax2.hist(outlet_C, bins=20, edgecolor='black', alpha=0.7)
ax2.axvline(C_ideal, color='r', linestyle='--', linewidth=2, label='Ideal mixed')
ax2.set_xlabel('Concentration (mol/m³)', fontsize=12)
ax2.set_ylabel('Frequency', fontsize=12)
ax2.set_title('Outlet Concentration Distribution', fontsize=13, fontweight='bold')
ax2.legend(fontsize=11)
ax2.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

## 7. Summary and Applications

### Key Findings:

1. **Flow Regime**: Laminar flow (Re << 1) - typical for microfluidics
2. **Transport Mechanism**: Advection-dominated (Pe >> 1) - diffusion is slow
3. **Mixing**: Limited transverse mixing due to low diffusion

### Biomolecular Applications:

1. **Protein Crystallization**: 
   - Controlled mixing of protein and precipitant solutions
   - Understanding concentration gradients for crystal nucleation

2. **Drug Delivery**:
   - Microfluidic synthesis of drug nanoparticles
   - Controlled release formulations

3. **Biosensors**:
   - Antibody-antigen binding kinetics
   - DNA hybridization studies

4. **Cell Studies**:
   - Chemical gradient generation for chemotaxis
   - Drug screening in concentration gradients

### Design Recommendations:

- **Increase mixing**: Add serpentine channels or obstacles
- **Reduce channel height**: Shorter diffusion distances
- **Increase residence time**: Longer channels or lower flow rates
- **Active mixing**: Electric fields, acoustic waves, or magnetic particles

## 8. Exercises

**Exercise 1**: Modify the code to simulate mixing of two proteins with different diffusion coefficients. How does the molecular weight affect mixing?

**Exercise 2**: Implement a reaction term: $r = k C_A C_B$ where proteins A and B bind. Plot the product concentration.

**Exercise 3**: Add obstacles in the channel to enhance mixing. Compare mixing efficiency.

**Exercise 4**: Investigate the effect of channel aspect ratio (Lx/Ly) on mixing performance.

**Exercise 5**: Calculate the pressure drop along the channel using the momentum equation.

In [ ]:
# Your code here for exercises


## References

1. Squires, T. M., & Quake, S. R. (2005). Microfluidics: Fluid physics at the nanoliter scale. *Reviews of Modern Physics*, 77(3), 977.

2. Stroock, A. D., et al. (2002). Chaotic mixer for microchannels. *Science*, 295(5555), 647-651.

3. DeMello, A. J. (2006). Control and detection of chemical reactions in microfluidic systems. *Nature*, 442(7101), 394-402.

4. Whitesides, G. M. (2006). The origins and the future of microfluidics. *Nature*, 442(7101), 368-373.